# 第24章 世界に公開する：ブラウザで動くあなたのLLM

HTMLとCSSとJavaScriptで、漱石GPTを誰でも使えるWebページにする

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：第13章の漱石GPTを8ビットに量子化してJSONに書き出し、JavaScriptで推論するWebページを作って、GitHub Pagesで世界に公開する（このサイトのトップページと同じもの）
- **身につく数学・理論**：推論だけなら、どこでも同じ計算ができる（言語を変えても同じ数式） ／ 量子化した重みの書き出し（第19章の実践） ／ ブラウザの中で動かす利点（サーバー代0円・入力が外に出ない） ／ 公開のいろいろな方法：GitHub Pages・Gradio・Ollama・Hugging Face ／ GGUFと量子化の名前の読み方 ／ 公開前のチェックリスト（ライセンス・安全・プライバシー）
- **新しく使うPython・PyTorch**：`files.download` でColabからファイルを取り出す ／ JavaScriptの読み方（Pythonとの対応表） ／ `python -m http.server` でローカル確認

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 2〜3時間 |
| 難易度 | ★★★☆☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> ついに、あなたのLLMを**世界に公開**します。第13章で育てた漱石GPTを、あなたが専門学校で学んだ **HTMLとCSS** と、少しの **JavaScript** でWebページにして、**GitHub Pages**（無料）で公開します。完成すると、スマートフォンからでも、友だちのパソコンからでも、URLを開くだけであなたのGPTが動きます。このサイトのトップページで動いている漱石GPTは、まさにこの章の方法で作ったものです。

> **🔗 これまでの復習とつながり：第3章・第12章・第13章・第19章とのつながり**
>
> - **第12〜13章（MiniGPTの設計と漱石GPTの重み）**：Pythonで組み立て、夏目漱石7作品で学習して Google Drive に保存した `soseki_gpt.pt`（約175万パラメータ）がそのまま主役になります。
> - **第3章（温度とsoftmaxサンプリング）**：`logits / temperature` → `softmax` → サイコロで1文字選ぶ計算が、そのまま `minigpt.js` の `sample()` 関数としてブラウザ上で動きます。
> - **第19章（量子化）**：32ビットの小数（`float32`）を8ビット整数（`int8`）に縮めるabsmax量子化を使うからこそ、重みファイルが約 **2.5 MB** に収まり、スマホ回線でも一瞬で開けます！

**ミナト**：やっとHTMLとCSSの出番だ！ でも、JavaScriptでGPTって動くんですか？ Pythonじゃないとダメなんじゃ……。

**ボンゴ先生**：GPTの正体は、掛け算と足し算と、expと平方根の組み合わせだっただろう？ そんなもの、どんな言語でも書ける。電卓でだって（何年かかるかは別として）計算できる。言語が変わっても、数式は変わらない。第15章でGPT-2の重みを自作コードに入れたのと同じで、今度は**別の言語**に重みを入れるんだ。

## 全体の流れ

1. **Colab**：第13章で保存したモデルを、Webで読める形（JSON）に書き出す
2. **パソコン**：HTML・CSS・JavaScriptのページを作り、手元で動作確認する
3. **GitHub**：ファイルをアップロードして、GitHub Pagesで公開する

## 手順1：モデルをJSONに書き出す（Colab）

> **✨ コツ：実行環境とリソースの目安**
>
> - **環境**：手順1（量子化書き出し）〜手順4（GitHub Pages公開）は**CPUだけで完走できます**（GPU不要、必要RAM約1GB未満、書き出しは数秒）。
> - **ファイルサイズ**：書き出す `soseki-gpt.json` は約2.5MBです。
> - **後半のGradio節を試す場合**：初回に `Qwen2.5-0.5B-Instruct`（約1GB）を取得します。GPUなら数秒、CPUでも動作しますが1回の返答に十数秒〜数十秒かかり、`device_map="auto"` には `accelerate` パッケージが必要です（推奨環境は `requirements-book.txt` を参照）。

Colabで新しいノートブックを作り、第13章でGoogleドライブに保存した `soseki_gpt.pt` を読み込みます。CPUのままで大丈夫です。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import os
ckpt = "/content/drive/MyDrive/tsukuru-llm/soseki_gpt.pt"
print("チェックポイントはある？", os.path.exists(ckpt))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
チェックポイントはある？ True
```

</details>

> **⚠️ つまずき注意：Falseだった人へ**
>
> 第13章を飛ばした人や、保存に失敗した人は、第13章に戻って学習と保存をしてから進むのがおすすめです。どうしても今すぐ試したい人は、この教科書のサイトに置いてある完成品のJSON（<https://sakunokyumei.github.io/llm-research-textbook/model/soseki-gpt.json>）をダウンロードして、手順2に進んでもかまいません。

書き出しの関数です。第19章のabsmax量子化を、重みの**行ごと**に行って、8ビットの整数にします。そして、JSONに入れられるように **base64**（バイト列を文字だけで表す方法）に変換します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import base64, json
import torch

def quantize_rows(w):
    w = w.float()
    if w.dim() == 1:                                   # バイアスやLayerNormはそのまま（float32）
        return {"shape": list(w.shape), "f32": base64.b64encode(w.numpy().tobytes()).decode()}
    scale = w.abs().amax(dim=1).clamp(min=1e-8) / 127  # 行ごとの縮尺
    q = torch.round(w / scale[:, None]).clamp(-127, 127).to(torch.int8)
    return {"shape": list(w.shape),
            "i8": base64.b64encode(q.numpy().tobytes()).decode(),
            "scale": base64.b64encode(scale.numpy().astype("float32").tobytes()).decode()}

ck = torch.load(ckpt, map_location="cpu", weights_only=True)   # 安全のためテンソルと基本データのみ読み込む（第17章）
weights = {name: quantize_rows(t) for name, t in ck["model"].items() if not name.endswith("tril")}
with open("soseki-gpt.json", "w", encoding="utf-8") as f:
    json.dump({"config": ck["config"], "chars": ck["chars"], "weights": weights}, f, ensure_ascii=False)
print("書き出したサイズ：", round(os.path.getsize("soseki-gpt.json") / 1e6, 1), "MB")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
書き出したサイズ： 2.5 MB
```

</details>

> **📘 基本情報とのつながり：base64**
>
> **base64** は、どんなバイト列も、英数字と `+` `/` の64種類の文字だけで表す方法です。3バイト（24ビット）を6ビットずつ4文字に分けて表すので、サイズは約1.33倍になります。メールに画像を添付するときにも使われています。基本情報の「ビットの切り分け」の練習問題そのものですね。

<details><summary><b>基礎の補足：重みの書き出しとサイズ圧縮のしくみ（PyTorchテンソル → 軽量JSON）</b>（クリックで開く）</summary>

なぜそのままJSONに書かず、`quantize_rows` でひと工夫しているのでしょうか？ 約175万パラメータのサイズを比べると一目瞭然です。

| 書き出し方法 | 1パラメータあたり | 175万個の目安サイズ | 特徴 |
|---|---|---|---|
| 素の小数リスト（`float` そのままJSON） | 約18文字（18 B） | **約 31 MB** | 桁数が長すぎてWeb配信には重すぎる |
| 小数3桁丸め（`round(x, 3)` のJSON） | 約6〜7文字（6.5 B） | **約 11 MB** | 実装は超簡単だが、まだやや大きい |
| 生の `float32` バイナリ（`.pt` 相当） | 4 B | **約 7.0 MB** | JSONに直接入れられない（base64化で約9.3 MB） |
| **行ごと `int8` 量子化 ＋ base64** | 1 B × 1.33 | **約 2.5 MB** | 精度を保ちつつ最小！スマホでも1秒未満で読める |

また、**1次元テンソル（バイアスやLayerNorm）は要素数がごく少なく精度に敏感なので `float32` のまま残し**、因果マスクの **`tril` は固定の下三角表で推論時に再現できるため除外** しています。

</details>

書き出したファイルを、自分のパソコンにダウンロードします。

☁️ **Google Colab専用セル**：ローカルPC・Kaggleでは不要または代替手順を使います。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
from google.colab import files
files.download("soseki-gpt.json")

#### 📝 確認問題（`ch24-q01` · Lv.2 · 重みの行ごと8ビット量子化・1次元層そのまま・tril除外の理由）

漱石GPTをブラウザ用のJSONに書き出す `quantize_rows` 関数で、①1次元テンソル（バイアスやLayerNorm）は量子化せず `float32` のまま残し、②名前が `tril` で終わるバッファは書き出しから除外している理由はどれですか？

- **a.** 1次元層は要素数が少なく精度劣化を避けたいから。`tril` は固定の下三角マスクで推論時に再生成でき保存がムダだから
- **b.** `tril` バッファをJSONに含めて書き出すと文字コード表が上書きされ、モデルが日本語ではなく英語しか話せなくなるから
- **c.** LayerNormやバイアスなどの1次元テンソルは学習終了後の推論計算では一度も使われず、JSONに書かなくても同じ出力になるから
- **d.** JavaScriptの言語仕様では8ビット整数（`Int8Array`）を一切扱えず、すべての重みを32ビット小数で持つしかないから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：175万パラメータの容量の大部分を占めるのは「1次元ベクトル」と「2次元行列」のどちらか、また `tril` は学習で変化する重みだったかを振り返りましょう。
- **ヒント 2**：1次元層（LayerNorm・バイアス）は小さくて精度に敏感である点と、`tril` が固定パターンの因果マスクである点の2つを結びつけてください。
- **正解**：`a`

**考え方**：モデル書き出しの鉄則は「①大部分の容量を占める2次元重みだけを効率よく量子化し、②小さくて敏感なLayerNorm/バイアスは高精度のまま残し、③学習パラメータではない固定マスク（`tril`）は省く」ことです。

1. 2次元行列（例：`[128, 512]`）→ 行ごとのabsmax 8ビット量子化でサイズを約1/4に圧縮。
1. 1次元ベクトル（例：`[128]`）→ 容量はほぼゼロなので `float32` のまま精度を保つ。
1. `tril` → 固定の0/1三角行列なのでJSONに書かず推論コード側で処理する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html#ex-ch24-q01

</details>

#### 📝 確認問題（`ch24-q02` · Lv.3 · float32・int8・base64のファイルサイズ計算）

パラメータ数がちょうど **200万個**（$2 \times 10^6$ 個）のモデルがあるとします。全パラメータを `float32`（1個4バイト）でそのまま保存すると **8.0 MB** になりますが、これを `int8`（1個1バイト）に量子化し、さらにJSONに埋め込むため **base64**（サイズが約 $4/3 \approx 1.333$ 倍）に変換すると、重み本体のサイズはおよそ何 MB になりますか？（小数第1位まで、誤差 $\pm 0.2$ MB まで正解）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：まず全パラメータを `int8`（1個1バイト）に圧縮したときの生バイト数をMB単位で求め、そこにbase64の倍率を掛けます。
- **ヒント 2**：200万個 × 1バイト ＝ `2.0 MB` です。これにbase64変換の倍率 $4/3 \approx 1.333$ を掛けて小数第1〜2位まで求めましょう。
- **正解**：`2.67` MB

**考え方**：float32（8.0 MB）→ int8（2.0 MB、1/4に圧縮）→ base64（2.0 × 4/3 ≈ 2.67 MB）。base64で1.33倍になっても、元のfloat32の約1/3まで軽くできています！

1. `int8` の生バイト数：$2\times 10^6 \times 1\text{ バイト} = 2.0\text{ MB}$。
1. base64は3バイトを4文字（4バイト）で表すので、サイズは $\dfrac{4}{3} \approx 1.333$ 倍になる。
1. $2.0 \times 1.333 \approx 2.67\text{ MB}$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html#ex-ch24-q02

</details>

*(この節の追加問題 `ch24-q03` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html で自動採点・復習できます)*

## 手順2：Webページを作る（パソコン）

パソコンに `soseki-gpt` という名前のフォルダを作り、次の3つのファイルを入れます。

```text
soseki-gpt/
├── index.html          ← これから作るページ
├── minigpt.js          ← GPTの計算をするJavaScript（教科書のものを使う）
└── soseki-gpt.json     ← 手順1でダウンロードしたモデル
```

`minigpt.js` は、このサイトのリポジトリの `site/minigpt.js` です。ブラウザで <https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/site/minigpt.js> を開き、右クリック→「名前を付けて保存」で保存してください。

### index.html

ここは、あなたの得意分野です。まずは次の内容で作り、後で好きなデザインに変えてください。

```html
<!doctype html>
<html lang="ja">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>わたしの漱石GPT</title>
  <style>
    body { font-family: system-ui, sans-serif; max-width: 640px; margin: 40px auto; padding: 0 16px;
           background: #faf7f0; color: #222; line-height: 1.8; }
    h1 { color: #b9501a; }
    input, button { font-size: 16px; padding: 8px 12px; border-radius: 8px; border: 1px solid #ccc; }
    button { background: #b9501a; color: white; border: none; cursor: pointer; }
    #out { white-space: pre-wrap; background: white; border-radius: 12px; padding: 16px; min-height: 8em; margin-top: 16px; }
  </style>
</head>
<body>
  <h1>わたしの漱石GPT</h1>
  <p>夏目漱石の7作品だけで学習した、約175万パラメータの小さなGPTです。ブラウザの中だけで動きます。</p>
  <input id="start" value="吾輩は">
  <button id="go" disabled>読み込み中…</button>
  <div id="out"></div>

  <script type="module">
    const go = document.getElementById("go");
    const out = document.getElementById("out");
    let model;
    try {                                   // Pythonの try / except と同じ（第23章）
      const { load } = await import("./minigpt.js");
      model = await load("./soseki-gpt.json");
    } catch (e) {
      go.textContent = "読み込み失敗";
      out.textContent = "モデルを読み込めませんでした：" + e.message +
        "\nファイル名（minigpt.js と soseki-gpt.json）と、http://localhost で開いているかを確かめてください。";
      throw e;
    }
    go.textContent = "続きを書かせる";
    go.disabled = false;
    go.addEventListener("click", async () => {
      const start = document.getElementById("start").value;
      out.textContent = start;
      go.disabled = true;
      try {
        await model.generate(start, 150, 0.8, (ch) => { out.textContent += ch; });
      } catch (e) {
        out.textContent += "\n[エラー] 生成に失敗しました：" + e.message;
      } finally {
        go.disabled = false;
      }
    });
  </script>
</body>
</html>
```

HTMLとCSSは見慣れた姿だと思います。新しいのは最後の `<script type="module">` の部分だけです。`try { … } catch (e) { … }` は、Pythonの `try` / `except`（第23章）と同じで、読み込みに失敗したら、画面に理由を表示します。

### minigpt.js の中身を読む

`minigpt.js` は約150行です。中で何をしているか、Pythonの対応する部分と比べてみましょう。第12章のコードと、ほぼ1対1で対応しています。

| JavaScript（minigpt.js） | Python（minigpt.py・第12章） | やっていること |
|---|---|---|
| `dequantize(w)` | — | 8ビットの整数 × 縮尺で、重みを小数に戻す（第19章） |
| `linear(W, b, x)` | `nn.Linear` | $y = Wx + b$。二重のforループで掛けて足す |
| `layerNorm(x, w, b)` | `nn.LayerNorm` | 平均を引いて標準偏差で割る（第10章） |
| `gelu(x)` | `nn.GELU` | 活性化関数（第9章） |
| `step(id, pos, cache)` の中のheadのループ | `Head.forward` | Q・K・V、内積、softmax、重みつき平均（第11章） |
| `cache[l][j].k.push(k)` | — | 過去のKeyとValueを取っておく **KVキャッシュ**（第16章） |
| `x[i] += a[i]` | `x = x + self.attn(...)` | 残差接続（第12章） |
| `sample(logits, temperature)` | `torch.multinomial` | 温度つきのサイコロ（第3章） |

> **🐍 Python・PyTorchのポイント：JavaScriptをPythonの目で読む**
>
> - `const` / `let`：変数を作る（`const` は後で入れ替えない変数）
> - `function f(x) { … return y; }`：`def f(x): … return y`
> - `for (let i = 0; i < n; i++) { … }`：`for i in range(n):`
> - `arr.map((v) => v * 2)`：`[v * 2 for v in arr]`
> - `Float32Array`：float32の数だけが並んだ配列（PyTorchのテンソルの、とても素朴な版）
> - `async` / `await`：「時間のかかる処理（ファイルの読み込みなど）を待つ」書き方
>
> インデント（字下げ）の代わりに `{ }` で範囲を表す、行の最後に `;` を付ける、くらいの違いで、考え方はPythonと同じです。

> **💡 理論のポイント：推論だけなら、どこでも動く（オンデバイス推論の魅力）**
>
> 学習には、自動微分（第7章）やAdam（第10章）、強力なGPUが必要でした。でも、**学習済みの重みで推論するだけ**なら、必要なのは掛け算と足し算と、exp・平方根などの簡単な関数だけです。だから、JavaScriptでも、スマートフォンでも、マイコンでも、原理的にはどこでも動かせます。
>
> 一般的なWebサービス（ChatGPTなど）は、ユーザーが文字を送ると、巨大なデータセンターのGPUサーバーへ通信が飛び、サーバー側で推論を計算して結果を返します。この方法だと、利用者が増えるほど高額なサーバー代（クラウド代）がかかります。
>
> それに対して、この章で作る方式は**オンデバイス推論（エッジ推論）**です。Webブラウザがモデルファイル（2.5MB）を最初に1回だけダウンロードし、そのあとは**アクセスした人のスマホやPCの頭脳（CPU）だけで推論を完結**させます。これには絶大なメリットがあります。
>
> - **サーバー代がほぼ0円**：静的ファイルを配信するだけなので、GitHub Pagesなどの無料サーバーで通常は料金がかかりません（ただしGitHub Pagesには月100GB程度の通信量目安があり、2.5MBのモデルならキャッシュなしで約4万回アクセス相当が目安です）
> - **入力が外に出ない（完全なプライバシー）**：ユーザーが打ち込んだ文章は、手元のブラウザの中だけで処理され、どこかの外部サーバーに送信されることがありません
> - **オフラインでも動く**：一度読み込んでしまえば、飛行機の中や電波のない地下鉄でも文章を生成できます
>
> ただし、数十億パラメータの大きなモデルは、端末でのダウンロードも計算も重くなります。175万パラメータ（2.5MB）くらいが、普通のブラウザで誰でも気軽に即座に試せる絶妙なサイズ感です。

<details><summary><b>基礎の補足：モデルサイズ vs ブラウザ初回ダウンロード待ち時間のトレードオフ</b>（クリックで開く）</summary>

| モデル規模 | 量子化後ファイル目安 | 5 MB/秒の回線での待ち時間 | ブラウザ静的公開の適性 |
|---|---|---|---|
| **漱石GPT（175万）** | **約 2.5 MB**（int8＋base64） | **約 0.5 秒** | ◎ 最適！スマホでも即起動・サーバー代0円・完全プライバシー |
| 小型モデル（5,000万） | 約 50 MB | 約 10 秒 | △ 実用ギリギリ（初回読み込みがやや長い） |
| Qwen2.5-0.5B（5億） | 約 500 MB（int8） | 約 100 秒 | × GitHub Pagesの100MB上限超え＆ブラウザ負担大 → GradioやOllamaへ |
| Qwen2.5-7B（70億） | 約 4.5 GB（Q4_K_M） | 約 15 分 | × ブラウザ不可 → ローカルの **Ollama** が最適 |

</details>

#### 📝 確認問題（`ch24-q04` · Lv.2 · minigpt.js と Python版 MiniGPT の部品対応の理解）

ブラウザ上でGPTを動かす `minigpt.js` の各関数と、これまでの章で作ったPythonコードの対応として**正しい**組み合わせはどれですか？

- **a.** `linear`＝線形層、`layerNorm`＝正規化、`cache`＝KVキャッシュ、`sample`＝温度つきサンプリング
- **b.** `linear`＝学習データの読み込み、`layerNorm`＝Adamオプティマイザ更新、`cache`＝勾配蓄積、`sample`＝誤差逆伝播
- **c.** JavaScriptには小数演算と `Math.exp` が無いため、`minigpt.js` の各関数はすべて外部Pythonサーバーへ計算を転送している
- **d.** `dequantize`＝HTMLのCSSレイアウト描画、`gelu`＝GitHub Pagesへの自動アップロード、`sample`＝ネットワーク通信制御

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：ブラウザ版 `minigpt.js` は「学習」ではなく「推論（順伝播＋KVキャッシュ＋サンプリング）」だけを行うプログラムです。
- **ヒント 2**：線形層・正規化層・過去K/Vの保存・温度による次トークン選択がそれぞれどの関数名に対応するか見比べてみましょう。
- **正解**：`a`

**考え方**：Python（PyTorch）で学んだ数式とブロック構造は、そのままJavaScriptの約150行に1対1で翻訳できます。プログラミング言語は変わってもLLMの数学的実体はまったく同じです。

1. 重みの復元：`dequantize`（第19章の量子化の逆）。
1. Transformer 1層：`layerNorm` → Attention（`cache`＝第16章KVキャッシュ）→ 残差接続 → `gelu` 付きFFN。
1. 次トークン選択：`sample(logits, temperature)`（第3章のsoftmax＋温度）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html#ex-ch24-q04

</details>

#### 📝 確認問題（`ch24-q05` · Lv.2 · ブラウザ推論（オンデバイス推論）の3大メリットと限界）

LLMをサーバー側ではなく、閲覧者のブラウザの中（オンデバイス）で動かす方式の**利点と欠点**の組み合わせとして正しいものはどれですか？

- **a.** 利点：1000億パラメータの巨大モデルでも0.01秒でダウンロード完了できる。欠点：閲覧者が増えるほど高額なGPUサーバー代がかかる
- **b.** 利点：サーバーGPU代0円・入力が外部送信されずプライバシー保護・オフライン動作可。欠点：端末負担が大きく巨大モデルに不向き
- **c.** 利点：重みファイル（JSON）を一切用意せずHTMLだけで文章生成できる。欠点：公開には常に有料のクレジットカード登録が必要になる
- **d.** 利点：利用者が入力した文章がすべて自動で公開リポジトリへ保存されて共有できる。欠点：パソコン専用でスマホからは一切開けない

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：ブラウザ推論では「誰の端末（CPU/メモリ）で計算しているか」「入力したテキストはどこへ送られるか」の2点を整理しましょう。
- **ヒント 2**：静的配信によるサーバー代0円・高いプライバシー保護という長所と、端末の通信量・メモリ上限によるモデルサイズの限界という短所の組み合わせを選びます。
- **正解**：`b`

**考え方**：オンデバイス推論は「計算コストとデータ保護を利用者側の端末で完結させる」設計です。小規模モデル（数MB〜数十MB）のデモやプライバシー重視アプリで絶大な力を発揮します。

1. サーバーの役割：`index.html`・`minigpt.js`・`soseki-gpt.json` を配るだけ（静的ホスティング＝無料）。
1. プライバシー：入力文はネットワークに1バイトも飛ばず、手元のブラウザ内だけで処理される。
1. 限界：ファイルサイズが大きすぎるとダウンロード待ち時間と端末メモリがボトルネックになる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html#ex-ch24-q05

</details>

*(この節の追加問題 `ch24-q06` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html で自動採点・復習できます)*

## 手順3：手元で動作確認する

作った `index.html` をダブルクリックで開いても、**モデルが読み込めません**（ブラウザの安全のための制限で、`file://` から別のファイルを `fetch` で読めないため）。簡単なWebサーバーを立てて確認します。ターミナル（Windowsならコマンドプロンプトや PowerShell）で、`soseki-gpt` フォルダに移動して、次を実行します。

```bash
cd soseki-gpt
python -m http.server 8000
```

> **✨ コツ：Windowsで、フォルダの場所でターミナルを開く**
>
> エクスプローラーで `soseki-gpt` フォルダを開き、上のアドレス欄（パスが表示されているところ）をクリックして `cmd` と入力し、<kbd>Enter</kbd> を押すと、そのフォルダでコマンドプロンプトが開きます。この場合、上の `cd soseki-gpt` は不要です。Macでは、Finderでフォルダを右クリック→「フォルダに新規ターミナル」（設定で有効にする必要がある場合あり）が便利です。

ブラウザで <http://localhost:8000/> を開き、「続きを書かせる」を押して、文章が出てくれば成功です。終わったら、ターミナルで <kbd>Ctrl</kbd> + <kbd>C</kbd> を押してサーバーを止めます。

> **⚠️ つまずき注意：「読み込み失敗」や「読み込み中…」のまま止まったら**
>
> - **ファイル名を確かめる**：2回ダウンロードすると `soseki-gpt (1).json` のような名前になっていることがあります。3つのファイルが `index.html`、`minigpt.js`、`soseki-gpt.json` という名前で、同じフォルダにあるか確かめましょう
> - **開き方を確かめる**：アドレス欄が `file://` で始まっていたら、上の手順でサーバーを立てて `http://localhost:8000/` で開き直します
> - **エラーの本文を見る**：<kbd>F12</kbd> キー（Macは <kbd>Cmd</kbd>+<kbd>Option</kbd>+<kbd>I</kbd>）で開発者ツールを開き、「Console（コンソール）」タブの赤い行を読みます。英語のエラーは、[エラー辞典](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)の質問テンプレートでAIに聞いてもOKです

> **✨ コツ：Pythonが入っていなかったら**
>
> 自分のパソコンにPythonが入っていない人は、[付録A](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html)の手順で入れるか、この手順を飛ばして手順4（GitHub Pages）で直接確かめてもかまいません。

#### 📝 確認問題（`ch24-q07` · Lv.2 · file:// でモデルが読み込めない理由と python -m http.server の役割）

パソコンに作った `index.html` をエクスプローラーから直接ダブルクリック（`file://...`）で開くと、`soseki-gpt.json` や `minigpt.js` の読み込みに失敗します。その理由と正しい対処法はどれですか？

- **a.** 同一オリジンポリシーで `file://` の読み込みが遮断されるため、ローカルHTTPサーバー越しに開く
- **b.** HTMLやJavaScriptはインターネット上の公開サーバーへアップロードしない限り、自分のパソコン内では絶対に動作確認できない仕様だから
- **c.** ダウンロード時に `soseki-gpt.json` の重みデータが破損してしまっているため、Colabへ戻って最初からモデルを学習し直すしかない
- **d.** `index.html` の中にあるエラー処理 `try / catch` をすべて削除すれば、ブラウザの制限が外れて `file://` のまま正常に動くようになる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：もし `file://` で開いた悪意あるHTMLページが、PC内の他のファイルを自由に `fetch` して読み取れたらセキュリティ上どうなるか想像しましょう。
- **ヒント 2**：ブラウザのセキュリティ制限（同一オリジンポリシー）を正しく満たして手元で開くための、Python標準モジュールのコマンドを思い出してください。
- **正解**：`a`

**考え方**：ブラウザはPC内の秘密ファイルが盗み読まれるのを防ぐため、`file://` からのモジュール読み込みや `fetch` を禁止しています。開発時はローカルHTTPサーバー越しに開くのがWeb開発の基本です。

1. ダブルクリック → URLが `file:///...` → ブラウザがセキュリティ上 `import`/`fetch` をブロック。
1. ターミナルで `python -m http.server 8000` を実行。
1. ブラウザで `http://localhost:8000/` を開くと正常に `minigpt.js` と `soseki-gpt.json` が読み込まれる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html#ex-ch24-q07

</details>

## 手順4：GitHub Pagesで世界に公開する

**GitHub** は、プログラムを保存・公開するための、世界最大のサービスです。**GitHub Pages** を使うと、置いたHTMLファイルを無料でWebサイトとして公開できます（このサイトも、GitHub Pagesで公開されています）。

1. <https://github.com/> でアカウントを作る（無料）。ユーザー名はURLの一部になるので、よく考えて
2. 右上の「＋」→「New repository」で、新しい置き場所（**リポジトリ**）を作る。名前は `soseki-gpt`、**Public**（公開）を選んで「Create repository」
3. 「uploading an existing file」のリンクを押し、`index.html`、`minigpt.js`、`soseki-gpt.json` の3つをドラッグ＆ドロップして、「Commit changes」
4. リポジトリの「Settings」→ 左の「Pages」→「Build and deployment」の「Branch」で `main` と `/ (root)` を選んで「Save」
5. 1〜2分待つと、`https://あなたのユーザー名.github.io/soseki-gpt/` で公開されます

**ボンゴ先生**：URLをスマートフォンで開いてごらん。キミが作ったGPTが、キミのポケットの中で動いている。サーバーもいらない、お金もかからない。友だちに送ったら、きっと驚くぞ。「これ、自分で作ったの？」って。ドコドコドン！

> **¥0 無料枠のコツ：GitHub Pages の注意**
>
> - 無料で使えるのは、**Public（公開）リポジトリ**です。中身は世界中から見えるので、パスワードやトークン（第17章）、個人情報は絶対に入れないでください
> - 1つのファイルは100MBまで、サイト全体は1GBまでが目安です。数百MB以上のモデルを置くのには向きません
> - 学習に使った青空文庫の作品は著作権が切れていますが、自分で集めたデータで学習したモデルを公開するときは、データの権利も確認しましょう

<details><summary><b>基礎の補足：静的サイト公開（GitHub Pages）の最終確認チェックリスト</b>（クリックで開く）</summary>

- [ ] `index.html`、`minigpt.js`、`soseki-gpt.json` の**3ファイルが同じルート階層**にあり、ファイル名に `(1)` などの余計な文字がない
- [ ] ローカルサーバー（`http://localhost:8000/`）でエラーなく生成できることを確認した
- [ ] リポジトリ内に API トークン・パスワード・個人情報が1行も入っていない
- [ ] 単一ファイルが 100 MB 以下（今回は約 2.5 MB）である
- [ ] Pages 設定で Branch `main`・フォルダ `/ (root)` を保存した

</details>

#### 📝 確認問題（`ch24-q08` · Lv.2 · GitHub Pages 静的公開の仕組みと注意点）

GitHub Pages（無料）で漱石GPTのWebページを公開するときに、**必ず守らなければならない注意点**はどれですか？

- **a.** 学習したPyTorchのモデルチェックポイント（5GB）を量子化や圧縮をせず生のままリポジトリへプッシュして公開すること
- **b.** Hugging Faceのアクセストークンや自分のAPIキーを `index.html` 内に直接書き込んでからリポジトリへアップロードすること
- **c.** 無料プランはPublic公開のため秘密情報や個人情報を絶対に含めず、1ファイル100MB・全体1GB以下を目安に収めること
- **d.** GitHub Pagesがトップページとして認識できるよう、エントリーポイントのファイル名を `index.html` ではなく `soseki.py` にすること

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：GitHubの無料Publicリポジトリは「世界中の誰でも全ファイルとコミット履歴を閲覧できる場所」である点を意識しましょう。
- **ヒント 2**：トークン・パスワードなどの秘密情報の扱いと、GitHubの1ファイルあたりのサイズ上限（100MB）の2点を確認してください。
- **正解**：`c`

**考え方**：GitHub Pagesは無料で非常に便利ですが、コード・JSON・コミット履歴が全世界に公開されるため、秘密情報の混入防止とファイルサイズ上限の遵守が必須です。

1. 配置するファイル：`index.html`・`minigpt.js`・`soseki-gpt.json`（合計約2.5MB）。
1. 秘密情報ゼロ：APIキー、パスワード、個人情報、他人の著作物の無断転載を入れない。
1. Settings → Pages → Branch `main` / `(root)` で公開URLが発行される。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html#ex-ch24-q08

</details>

## そのほかの公開・共有の方法

GitHub Pagesは小さなモデル向けです。Qwenのような大きめのモデルを人に使ってもらうには、ほかの方法もあります。

### Colabで一時的に公開する：Gradio

**Gradio** は、Pythonだけでチャット画面などのWebアプリを作れるライブラリです。Colabで `share=True` にすると、一時的な公開URL（最長1週間ほど有効）が発行されます。第18〜19章のボンゴ先生モデルを、友だちに試してもらうのにぴったりです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
!pip install -q gradio
import gradio as gr
from transformers import pipeline

bot = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def text_of(content):
    if isinstance(content, list):          # 新しいGradioでは、文字以外の部品のリストのこともある
        return "".join(c.get("text", "") for c in content if isinstance(c, dict))
    return content

def reply(message, history):
    messages = [{"role": h["role"], "content": text_of(h["content"])} for h in history]
    messages.append({"role": "user", "content": text_of(message)})
    out = bot(messages, max_new_tokens=200)
    return out[0]["generated_text"][-1]["content"]

demo = gr.ChatInterface(reply, title="ボンゴ先生に質問しよう")
print(reply("こんにちは！", []))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
こんにちは！何かお手伝いできることはありますか？
```

</details>

☁️ **Google Colab専用セル**：ローカルPC・Kaggleでは不要または代替手順を使います。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
demo.launch(share=True)

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
* Running on public URL: https://xxxxxxxxxxxxxxxx.gradio.live
```

</details>

> **⚠️ つまずき注意：Gradioの版の違い**
>
> Gradioは版によって書き方が少しずつ変わります。古い解説記事に `gr.ChatInterface(…, type="messages")` と書いてあっても、新しい版（6系）ではこの引数がなくなっていて、エラーになります。エラーが出たら、まずGradioの版（`print(gr.__version__)`）と、公式ドキュメントを確認しましょう。

`model=` の部分を、第18章で保存したフォルダ（`/content/drive/MyDrive/tsukuru-llm/bongo-sft`）に変えれば、あなたのボンゴ先生が公開されます（このノートブックで先に `drive.mount` のセルを実行して、ドライブをつないでおきましょう）。Colabのセッションが切れると、URLも使えなくなります。

### 自分のパソコンで動かす：Ollama

**Ollama**（オラマ）は、LLMを自分のパソコンで簡単に動かせる無料のアプリです。<https://ollama.com/> からインストールして、ターミナルで次を実行するだけで、Qwen2.5-0.5Bと会話できます（初回はダウンロードに時間がかかります）。

```bash
ollama run qwen2.5:0.5b
```

インターネットにつながっていなくても動き、入力した内容が外に送られることもありません。少し性能の良いパソコンなら、`qwen2.5:7b` なども動きます。

> **💡 理論のポイント：GGUFと、量子化の名前**
>
> OllamaやLM Studioなどのアプリは、**GGUF** という形式のモデルファイルを使います。llama.cpp というC++で書かれた高速な推論プログラムのための形式で、量子化した重みをまとめて1つのファイルにしたものです。
>
> ファイル名についている `Q4_K_M` や `Q8_0` は、量子化の方法の名前です。`Q4` は約4ビット、`Q8` は約8ビット。数字が小さいほどファイルは小さく、性能は少し下がります。迷ったら `Q4_K_M` がよく使われる、バランスの良い選択です。第19章で手計算した量子化が、ここで使われているわけです。

### Hugging Faceで共有する

学習したモデルやLoRAのアダプタは、Hugging Faceにアップロードして共有できます（第18章の「もっと深く」）。アプリを公開する **Spaces** は、2026年10月時点で、無料アカウントでは種類が限られています（CPUで動くGradioアプリの新規作成には有料プランが必要で、無料では「ZeroGPU」を使うSpaceが2つまで、または**静的なHTMLのSpace**が使えます）。静的なSpaceなら、この章で作ったページ（3つのファイル）をそのまま置いて公開することもできます。サービスの条件はよく変わるので、使う前に公式ドキュメントで確認しましょう。

#### 📝 確認問題（`ch24-q09` · Lv.2 · GitHub Pages・Gradio・Ollama・Hugging Faceの使い分け）

次の3つの目的に対して、最も適した公開・実行方法の組み合わせはどれですか？
① 約2.5MBの漱石GPTを無料・常時公開で誰でもスマホから試せるようにしたい
② Colab上のボンゴ先生モデル（Qwen2.5-0.5B＋LoRA）を、授業発表の30分間だけ友だちにWebチャットで試してもらいたい
③ 自分のPC上で、インターネット未接続のままQwen2.5-7Bと会話したい

- **a.** ① Ollama（ローカル実行）、② GitHub Pages（ブラウザ推論）、③ Gradio（Colabで `share=True`）
- **b.** ①〜③のすべてについて、`index.html` を `file://` でダブルクリックして開いたURLを友だちへ共有する
- **c.** ①〜③のすべてのモデルと用途を、サイズに関係なくGitHub Pages上の静的ブラウザ推論だけでまとめて公開する
- **d.** ① GitHub Pages（ブラウザ推論）、② Gradio（Colabで `share=True`）、③ Ollama（ローカル実行）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：3つの要件を「モデルサイズ（2.5MB / 0.5B / 7B）」「公開期間（常時 / 30分）」「オフライン要件」の3軸で整理しましょう。
- **ヒント 2**：数MBの常設デモ＝静的ホスティング、Colab上の一時共有＝Gradio、手元PCのオフライン7B＝Ollama の対応を当てはめます。
- **正解**：`d`

**考え方**：LLMの公開に「万能の1択」はなく、モデルの大きさ・予算（0円）・プライバシー・公開期間に応じて最適な手段を選び分けます。

1. 数MB級＋無料常時公開 → GitHub Pages（または静的HF Space）のブラウザ推論。
1. 0.5B〜7B級＋短時間のデモ共有 → Colab＋Gradio `share=True`（セッション終了まで有効）。
1. 手元PCでオフライン・高速常用 → Ollama（GGUF形式）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html#ex-ch24-q09

</details>

#### 📝 確認問題（`ch24-q10` · Lv.3 · GGUF形式と量子化名（Q4_K_M / Q8_0）の読み方）

Ollamaやllama.cppで使われるGGUFモデルファイルで、同じ7Bモデルの `Q4_K_M` 版と `Q8_0` 版を比べたときの正しい説明はどれですか？

- **a.** `Q4_K_M` は圧縮テーブルの容量が大きいため、同じ7Bモデルでも常に `Q8_0` よりファイルサイズが2倍以上大きくなる
- **b.** GGUFはWebページの見た目やボタン配置を記述するためのHTML/CSS系のマークアップ言語で、重みデータは保存できない
- **c.** `Q4_K_M` は約4ビット量子化で軽量・高速なバランス型、`Q8_0` は約8ビット量子化でサイズは約2倍だが元の精度により近い
- **d.** `Q4_K_M` は総パラメータ数がちょうど4個だけの超小型モデルで、`Q8_0` はパラメータ数が8個に倍増した上位モデルである

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`Q4` と `Q8` の数字が「1パラメータあたりのビット数」を表していることを思い出しましょう。
- **ヒント 2**：ビット数が半分になるとファイルサイズやメモリ消費量、そして元の浮動小数点精度との近さがどう変わるか比べてください。
- **正解**：`c`

**考え方**：ローカルLLM選びでは、自分のPCのRAM/VRAM容量に合わせて量子化レベルを選びます。迷ったときの定番が容量と品質のバランスに優れた `Q4_K_M` です。

1. 7Bモデルの目安：FP16なら約14GB、`Q8_0`（約8bit）なら約7〜8GB、`Q4_K_M`（約4bit）なら約4〜5GB。
1. メモリに余裕があり精度重視 → `Q8_0`。
1. 一般PCで軽快に動かしたい → `Q4_K_M`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html#ex-ch24-q10

</details>

## 公開前のチェックリスト

> **⚠️ つまずき注意：公開する前に確かめること**
>
> 1. **ライセンス**：元のモデル（Qwen2.5はApache 2.0）と、学習データのライセンスを守っているか。モデルカードやページに書いたか
> 2. **秘密の情報**：トークン、パスワード、個人情報が、コードやデータに入っていないか
> 3. **出力の注意書き**：「AIが生成した文章で、内容が正しいとは限りません」と書いたか
> 4. **悪用の可能性**：誰かを傷つける文章を簡単に作れてしまわないか
> 5. **お金**：クレジットカードの登録が必要なサービスを、知らないうちに使っていないか（0円の原則）

> **🔬 やってみよう：自分のページをデザインしよう**
>
> `index.html` のCSSを書き換えて、自分だけのデザインにしましょう。和紙風の背景、縦書き（`writing-mode: vertical-rl`）、温度のスライダー（`<input type="range">`）、生成した文章をX（旧Twitter）で共有するボタン……。HTMLとCSSの腕の見せどころです。完成したら、URLを第26章の卒業制作のポートフォリオに載せましょう。

#### 📝 確認問題（`ch24-q11` · Lv.3 · モデル・Webアプリ公開前の5つの必須チェック項目）

自作のLLMデモページをインターネットに公開する直前のチェックリストとして、**最も不適切な判断**はどれですか？

- **a.** 元モデル（例：Qwen2.5のApache 2.0）と学習データ（例：青空文庫）のライセンスを確認してページやモデルカードに明記する
- **b.** 公開用コードやJSONファイルの中にAPIトークン・パスワード・個人情報が混入していないか事前に最終確認する
- **c.** 個人サイトだからと元モデルのライセンス表示やAI生成の注意書きを省き、学習データの著作権も確認せず公開する
- **d.** 利用者がAIの誤情報を鵜呑みにしないよう注意書きを載せ、悪用の可能性や意図しない課金サービスを使っていないか確認する

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：公開前の5つの必須チェック項目（ライセンス・秘密情報・注意書き・悪用の可能性・課金有無）を思い出しましょう。
- **ヒント 2**：この問題は「**最も不適切な判断**」を選ぶ形式です。5つのチェックを無視している選択肢を探してください。
- **正解**：`c`

**考え方**：個人制作であってもインターネットに公開した瞬間から世界中がアクセス可能になります。権利関係・プライバシー・安全・注意書き・費用の5点は公開前に必ず確認します。

1. ①ライセンス：元モデルと学習データの規約を守り明記する。
1. ②秘密情報：トークン・個人情報ゼロを徹底。
1. ③注意書き：「AIが生成した文章で正確とは限らない」を明示。
1. ④悪用の可能性と⑤予期せぬ課金がないかを確認する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html#ex-ch24-q11

</details>

#### 📝 確認問題（`ch24-q12` · Lv.5 · 【総合】用途・モデル規模・プライバシー・予算0円に応じた公開方式の設計）

次の2つのケースについて、それぞれどの公開・実行方式（GitHub Pagesブラウザ推論／Colab＋Gradio／ローカルOllama）を選ぶべきか、①ファイルサイズと待ち時間、②プライバシー、③サーバー代（0円制約）、④公開前のチェック事項の4観点から理由を説明してください。
- **ケースA**：夏目漱石風のミニGPT（175万パラメータ）を、自分のポートフォリオサイトに常設して誰でもURLから試せるようにしたい。
- **ケースB**：自分の日記や非公開の個人メモを要約・整理するAI（Qwen2.5-7B）を、外部サーバーに一切データを送らず毎日使いたい。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：2つのケースを「モデルサイズ（約2.5MB vs 約4〜5GB）」「公開形態（常設URL公開 vs 個人の機密データ処理）」の違いで整理しましょう。
- **ヒント 2**：ケースAには軽量・常設・0円を満たすブラウザ静的公開（＋公開前チェック）、ケースBには7Bのサイズと外部送信不可を満たすローカル実行方式を選び、4観点で理由をまとめます。
- **解答の目安**：**ケースA**は **GitHub Pagesでのブラウザ推論** が最適です。175万パラメータを8ビット量子化＋base64化すれば約2.5MBに収まり、スマホ回線でも1秒未満で読み込めます。静的ファイル配信のため常設してもサーバー代は完全0円、利用者の入力文も外に出ずプライバシーが守られます。公開前には学習データ（青空文庫の著作権切れ）の明記、秘密情報混入チェック、「AI生成の注意書き」の記載を行います。**ケースB**は **ローカルのOllama（GGUF `Q4_K_M` など）** が最適です。7Bモデルは4ビット量子化でも約4〜5GBあり、ブラウザ静的公開ではサイズ上限・待ち時間・メモリとも現実的ではありません。一方Colab＋Gradioでは非公開の日記が外部クラウド（Google/Gradio）に送られ、セッションも切れます。Ollamaなら自分のPC内だけで完全オフライン・無料・プライバシー100%で毎日安定して動かせます。

**考え方**：モデルサイズ（2.5MB vs 4.5GB）・利用形態（不特定多数への常設公開 vs 個人の機密データ処理）を見極めることで、無駄なコストやプライバシー事故を防げる最適アーキテクチャを選べます。

1. ケースA：2.5MBで軽量・常設・無料 → GitHub Pages（ブラウザ推論）。公開前チェックリストを確認。
1. ケースB：7B（約4.5GB）はブラウザ不可、機密日記なので外部送信も不可 → ローカルOllama（GGUF量子化）。
1. 両方のトレードオフ（サイズ・待ち時間・プライバシー・0円制約）を比較して説明する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html#ex-ch24-q12

</details>

**✅ できたらチェック**

- [ ] 第13章のモデルを、8ビットに量子化してJSONに書き出した
- [ ] `minigpt.js` の主な関数が、Pythonのどの部品に対応するか説明できる
- [ ] 手元のWebサーバーで、自分の漱石GPTを動かした
- [ ] GitHub Pagesで公開し、スマートフォンからも開けた
- [ ] Gradio・Ollama・Hugging Faceの使い分けを説明できる
- [ ] 公開前のチェックリストを確認した

> **📝 まとめ**
>
> - 推論は掛け算と足し算の組み合わせなので、**どの言語でも**書ける。JavaScriptならブラウザで動く
> - ブラウザで動かすと、**サーバー代0円**・**入力が外に出ない**
> - 重みは**量子化**して**base64**でJSONに入れ、`minigpt.js` で読み込む
> - **GitHub Pages**で無料公開。大きめのモデルは **Gradio**（一時公開）や **Ollama**（ローカル）で
> - 公開前に、ライセンス・秘密情報・注意書き・悪用・お金をチェック

**ボンゴ先生**：第6部、完走だ！ キミのLLMは、もう世界のどこからでも動かせる。残るは第7部、研究の入口だ。最新のLLMが使っている部品（RoPE、RMSNorm、SwiGLU、MoE）を、キミのGPTに移植して、本当に良くなるのかを**実験で確かめる**。そして最後に、卒業制作と、2030年までの地図を描こう。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html
- **次の章**：[第25章 最新LLMの部品を移植する](https://sakunokyumei.github.io/llm-research-textbook/25-modern.html)